# Rich single-sample graph-state visualization

This notebook studies **one response graph at a time**. The primary response-node representation is a 32-D deterministic graph state built from incoming prompt/history routing, sparsity, concentration, locality, and early/middle/late layer routing. t-SNE is fitted without hallucination labels; labels are added only afterwards for diagnostic coloring.

The main phase view distinguishes `far normal`, `pre-error`, `hallucination`, and `post-error`. A separate all-token source-role view includes prompt tokens; because it summarizes how a token is used by later response queries, it is descriptive rather than causal.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from rich_visualization import RichSampleVisualizer

DATA_ROOT = Path(
    "/share/home/tm902089733300000/a903202310/lys/data/RAGTruth/"
    "model_traces/llama31_8b/test"
)
OUTPUT_ROOT = REPO_ROOT / "outputs" / "single_sample_rich"

ERROR_INDEX = 0
ERROR_SAMPLE_ID = None  # e.g. "10071"; None uses ERROR_INDEX
PRE_WINDOW = 10
POST_WINDOW = 10
HEATMAP_RADIUS = 18


In [ ]:
viewer = RichSampleVisualizer(
    DATA_ROOT,
    output_root=OUTPUT_ROOT,
    device="cpu",
    verify_hashes=False,
    random_state=0,
)

print("total samples:", len(viewer.dataset))
print("hallucinated samples:", len(viewer.error_sample_ids))
print("fully correct samples:", len(viewer.correct_sample_ids))
viewer.list_errors(limit=10)


## Select one hallucinated sample

Set `ERROR_SAMPLE_ID` above for an exact sample, or change `ERROR_INDEX`.


In [ ]:
sample_id = (
    str(ERROR_SAMPLE_ID)
    if ERROR_SAMPLE_ID is not None
    else viewer.error_sample_ids[ERROR_INDEX]
)
print("selected sample:", sample_id)
print("positive runs:", viewer.labels.positive_runs(sample_id))
print("metadata:", viewer.dataset[sample_id].metadata)


## One-call rich diagnostic bundle

This generates the response-node phase projection, multi-perplexity stability view, prompt/response source-role projection, onset-centered feature heatmap, pre-error→hallucination feature-change plot, and a joint projection with a matched correct control.


In [ ]:
result = viewer.visualize(
    sample_id,
    pre_window=PRE_WINDOW,
    post_window=POST_WINDOW,
    heatmap_radius=HEATMAP_RADIUS,
    include_control=True,
)
result["metadata"]


## Display the saved figures


In [ ]:
from IPython.display import Image, display

sample_output = OUTPUT_ROOT / sample_id
for filename in (
    "rich_response_tsne.png",
    "projection_stability.png",
    "source_role_tsne.png",
    "rich_feature_heatmap.png",
    "rich_feature_differences.png",
    "matched_control_joint_tsne.png",
):
    path = sample_output / filename
    if path.is_file():
        print(filename)
        display(Image(filename=str(path)))


## Quantitative checks in the original feature space

These metrics do **not** use the t-SNE coordinates. A positive silhouette value indicates that hallucination vs non-hallucination nodes are better separated in the robust-scaled 32-D space; the centroid distance measures how far the hallucination segment moves from the immediately preceding pre-error segment.


In [ ]:
viewer.separation_metrics(
    sample_id,
    pre_window=PRE_WINDOW,
    post_window=POST_WINDOW,
)


## Inspect the largest structural changes

Positive values mean the hallucination span is larger than the preceding pre-error window after within-sample standardization; negative values mean it is smaller.


In [ ]:
differences = viewer.feature_differences(
    sample_id,
    pre_window=PRE_WINDOW,
)
order = abs(differences["difference"]).argsort()[::-1]
[
    (str(differences["feature_names"][i]), float(differences["difference"][i]))
    for i in order[:20]
]
